# Model II

Biblioteki

In [1]:
from datetime import datetime
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

import time
from model import MultiInstrumentCNN
from trening import get_device, train_model
from przygotowanie_danych import (INSTRUMENTS, create_datasets, create_loaders)

### Konfiguracja eksperymentu

In [3]:
dataset = "1_1"
model_name = "CNN2"

epochs = 100
patience = 10
batch_size = 8
learning_rate = 0.001

dropout2 = 0.1
dropout3 = 0.3

threshold = 0.8

results_file = Path("wyniki/modele.csv")

Sprawdzenie, czy eksperyment był już wykonywany

In [4]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout 2"] == dropout2) &
        (results["Dropout 3"] == dropout3)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymanie programu."
        )

print("Nowa kombinacja. Kontynuacja wykonywania kodu.")


Nowa kombinacja. Kontynuacja wykonywania kodu.


### Nadanie identyfikatora eksperymentu

In [5]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 1
Plik wag: 1.pth


### Przygotowanie danych

In [6]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=batch_size
)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Wczytuję spektrogramy z: D:\spectrograms\1_1
Train: 7758
Validation: 1658
Test: 1664


### Inicjalizacja modelu

In [7]:
device = get_device()
print(f"Urządzenie: {device}")

model = MultiInstrumentCNN(num_instruments=len(INSTRUMENTS), dropout2=dropout2, dropout3=dropout3).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Urządzenie: cuda


### Trening sieci

In [8]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.6141 | Val loss: 0.5639
Epoka 2/100 | Train loss: 0.5448 | Val loss: 0.5190
Epoka 3/100 | Train loss: 0.5222 | Val loss: 0.4911
Epoka 4/100 | Train loss: 0.5044 | Val loss: 0.4688
Epoka 5/100 | Train loss: 0.4810 | Val loss: 0.4600
Epoka 6/100 | Train loss: 0.4659 | Val loss: 0.4472
Epoka 7/100 | Train loss: 0.4542 | Val loss: 0.4333
Epoka 8/100 | Train loss: 0.4423 | Val loss: 0.4256
Epoka 9/100 | Train loss: 0.4334 | Val loss: 0.4217
Epoka 10/100 | Train loss: 0.4271 | Val loss: 0.4069
Epoka 11/100 | Train loss: 0.4131 | Val loss: 0.3992
Epoka 12/100 | Train loss: 0.4052 | Val loss: 0.3968
Epoka 13/100 | Train loss: 0.3933 | Val loss: 0.3675
Epoka 14/100 | Train loss: 0.3871 | Val loss: 0.3709
Epoka 15/100 | Train loss: 0.3803 | Val loss: 0.3595
Epoka 16/100 | Train loss: 0.3734 | Val loss: 0.3542
Epoka 17/100 | Train loss: 0.3662 | Val loss: 0.3472
Epoka 18/100 | Train loss: 0.3599 | Val loss: 0.3680
Epoka 19/100 | Train loss: 0.3542 | Val loss: 0.3505
Ep

### Zapis informacji o eksperymencie

In [10]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout 2": dropout2,
    "Dropout 3": dropout3,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")


Informacje o eksperymencie zapisano w: wyniki\modele.csv
Historia uczenia: history_1.csv
